In [2]:
import pandas as pd

df_2024 = pd.read_csv("../data/raw/atp_matches_2024.csv")
df_2025 = pd.read_csv("../data/raw/atp_matches_2025.csv")

print("2024 matches:", len(df_2024))
print("2025 matches:", len(df_2025))
print("Total matches:", len(df_2024) + len(df_2025))

2024 matches: 3076
2025 matches: 2944
Total matches: 6020


In [3]:
print(df_2024.shape)
print(df_2025.shape)

print("\nColumns:")
print(df_2024.columns.tolist())

(3076, 49)
(2944, 49)

Columns:
['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points']


In [4]:
df = pd.concat([df_2024, df_2025])

print(df["surface"].value_counts())

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
surface
Hard     3596
Clay     1800
Grass     624
Name: count, dtype: int64


In [5]:
df = pd.concat([df_2024, df_2025])

print(df["tourney_level"].value_counts())

tourney_level
A    2898
M    1470
G    1016
D     512
O      64
F      60
Name: count, dtype: int64


In [6]:
print(df[["winner_rank", "loser_rank"]].describe())

       winner_rank   loser_rank
count  5991.000000  5924.000000
mean     82.520114   110.314821
std     148.278683   178.864074
min       1.000000     1.000000
25%      17.000000    35.000000
50%      46.000000    65.000000
75%      87.000000   109.000000
max    2068.000000  2171.000000


In [7]:
df["rank_difference"] = df["loser_rank"] - df["winner_rank"]

print(df["rank_difference"].describe())

count    5915.000000
mean       32.022992
std       169.594476
min     -1655.000000
25%       -20.500000
50%        19.000000
75%        61.000000
max      1922.000000
Name: rank_difference, dtype: float64


In [8]:
print(df["round"].value_counts())

round
R32     1728
R16      992
R128     896
R64      896
RR       614
QF       503
SF       260
F        130
BR         1
Name: count, dtype: int64


In [9]:
missing_values = df.isnull().sum()

print(
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
)

winner_entry          5063
loser_entry           4598
loser_seed            4533
winner_seed           3498
minutes                636
w_SvGms                320
l_SvGms                320
w_ace                  319
w_svpt                 319
w_1stIn                319
w_1stWon               319
w_2ndWon               319
w_bpFaced              319
l_ace                  319
w_bpSaved              319
w_df                   319
l_2ndWon               319
l_bpSaved              319
l_bpFaced              319
l_df                   319
l_svpt                 319
l_1stIn                319
l_1stWon               319
loser_ht               144
rank_difference        105
loser_rank_points       96
loser_rank              96
winner_ht               74
winner_rank             29
winner_rank_points      29
loser_age                6
winner_age               4
dtype: int64


In [10]:
higher_ranked_wins = (
    df["winner_rank"] < df["loser_rank"]
).mean()

print(
    f"Higher ranked player wins: {higher_ranked_wins:.2%}"
)

Higher ranked player wins: 62.94%


In [11]:
# V1 Engineered Features

df["rank_difference"] = df["loser_rank"] - df["winner_rank"]

df["rank_points_difference"] = (
    df["winner_rank_points"] -
    df["loser_rank_points"]
)

df["age_difference"] = (
    df["winner_age"] -
    df["loser_age"]
)

df["height_difference"] = (
    df["winner_ht"] -
    df["loser_ht"]
)

print("Features created successfully")

Features created successfully


In [12]:
print(df[
    [
        "rank_difference",
        "rank_points_difference",
        "age_difference",
        "height_difference"
    ]
].describe())

       rank_difference  rank_points_difference  age_difference  \
count      5915.000000             5915.000000     6010.000000   
mean         32.022992              740.655959       -0.361681   
std         169.594476             2229.428617        6.042061   
min       -1655.000000            -9857.000000      -23.200000   
25%         -20.500000             -269.000000       -4.200000   
50%          19.000000              271.000000       -0.500000   
75%          61.000000             1255.500000        3.500000   
max        1922.000000            11600.000000       22.200000   

       height_difference  
count        5835.000000  
mean            0.989032  
std             9.510214  
min          -112.000000  
25%            -5.000000  
50%             0.000000  
75%             8.000000  
max           130.000000  


In [13]:
df[
    abs(df["height_difference"]) > 50
][
    [
        "winner_name",
        "loser_name",
        "winner_ht",
        "loser_ht",
        "height_difference"
    ]
]

,winner_name,loser_name,winner_ht,loser_ht,height_difference
2916,Christopher Eubanks,Viacheslav Bielinskyi,201.0,71.0,130.0
2901,Viacheslav Bielinskyi,Peter Bertran,71.0,183.0,-112.0


In [14]:
print(
    sorted(
        df["winner_ht"]
        .dropna()
        .unique()
    )[:30]
)

[np.float64(71.0), np.float64(170.0), np.float64(173.0), np.float64(175.0), np.float64(178.0), np.float64(180.0), np.float64(183.0), np.float64(185.0), np.float64(187.0), np.float64(188.0), np.float64(191.0), np.float64(193.0), np.float64(196.0), np.float64(198.0), np.float64(201.0), np.float64(203.0), np.float64(206.0), np.float64(211.0)]


In [15]:
print(
    sorted(
        df["winner_ht"]
        .dropna()
        .unique()
    )[-30:]
)

[np.float64(71.0), np.float64(170.0), np.float64(173.0), np.float64(175.0), np.float64(178.0), np.float64(180.0), np.float64(183.0), np.float64(185.0), np.float64(187.0), np.float64(188.0), np.float64(191.0), np.float64(193.0), np.float64(196.0), np.float64(198.0), np.float64(201.0), np.float64(203.0), np.float64(206.0), np.float64(211.0)]
